# Download an OCR example dataset

Use the [energy-meter dataset](https://huggingface.co/datasets/henrik-dra/energy-meter) to try this repository on real images. It contains meter readings, so its scores do not measure license-plate or price-tag accuracy. The dataset files are about 361 MB.

Install the downloader dependencies in the notebook kernel with `%pip install huggingface_hub pyarrow Pillow`.

In [ ]:
from pathlib import Path

DOWNLOAD = False
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'README.md').is_file() and (p / 'notebooks').is_dir())
OUTPUT = ROOT / 'dataset/energy-meter'
DATASET = 'henrik-dra/energy-meter'
REVISION = 'b21598cf9109cdfb68e3f16276b78ce3a5d55639'


Set `DOWNLOAD = True` and run the code cells. The converter saves the published test split as `val` and removes any training rows whose image pixels match a validation image. Labels are written verbatim as UTF-8.

In [ ]:
if DOWNLOAD:
    from hashlib import sha256
    from io import BytesIO
    from tempfile import TemporaryDirectory
    from huggingface_hub import hf_hub_download
    from PIL import Image
    import pyarrow.parquet as pq

    if OUTPUT.exists():
        raise FileExistsError(f'Choose a new OUTPUT directory: {OUTPUT}')
    OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(dir=OUTPUT.parent) as staging:
        staging = Path(staging)
        dataset = staging / 'dataset'
        counts = {}
        validation_pixels = set()
        skipped = 0
        for source, dest in [('test', 'val'), ('train', 'train')]:
            parquet = hf_hub_download(DATASET, f'data/{source}-00000-of-00001.parquet',
                                      repo_type='dataset', revision=REVISION)
            target = dataset / dest
            target.mkdir(parents=True)
            count = 0
            for batch in pq.ParquetFile(parquet).iter_batches(columns=['image', 'label'], batch_size=4):
                for row in batch.to_pylist():
                    stem = target / f'{count:05d}'
                    with Image.open(BytesIO(row['image']['bytes'])) as image:
                        rgb = image.convert('RGB')
                        digest = sha256(str(rgb.size).encode() + rgb.tobytes()).digest()
                        if dest == 'val':
                            validation_pixels.add(digest)
                        elif digest in validation_pixels:
                            skipped += 1
                            continue
                        rgb.save(stem.with_suffix('.png'), compress_level=1)
                    stem.with_suffix('.txt').write_text(row['label'], encoding='utf-8')
                    count += 1
            counts[dest] = count
        dataset.rename(OUTPUT)
    print(OUTPUT.resolve(), counts, f'Cross-split duplicates removed from training: {skipped}')
else:
    print('Set DOWNLOAD = True to fetch and convert the example.')


## Run the example

[1_dataset_preparation.ipynb](1_dataset_preparation.ipynb). Set `WATEREC_CONFIG=configs/energy_meter.yaml` in the kernel environment (for example, `%env WATEREC_CONFIG=configs/energy_meter.yaml`) before running dataset preparation. Run the numbered notebooks in the same kernel environment.